### Wczytanie danych (i bibliotek)

# 1. Eksploracyjna analiza danych

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve, mean_squared_error, 
                              r2_score, classification_report, confusion_matrix)
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

import warnings
warnings.filterwarnings("ignore")

### Wczytanie i zrozumienie danych

In [ ]:
df = pd.read_csv('Hotel Reservations.csv')
print(f"Liczba wierszy: {df.shape[0]}, liczba kolumn: {df.shape[1]}")
df.head()

### Typy danych i zmiennych

In [ ]:
df.info()

In [ ]:
print("Zmienne numeryczne")
df.describe()

In [ ]:
print("Zmienne opisowe")
df.describe(include='object')

In [ ]:
df.market_segment_type.unique()

# TODO 
### zrozumienie zmiennych? rozklady cech, korelacje, wykresy itd

### Sprawdzenie brakujących danych

In [ ]:
print("Liczba brakujących wartości")
df.isnull().sum()

In [ ]:
print("Liczba duplikatów")
df.duplicated().sum()

In [ ]:
df['booking_status'].value_counts(normalize=True) # Można tym sprawdzić sobie stosunek cancelled/not cancelled

### Hipotezy

Analizująć kolejne kolumny z ramki danych możemy postawić następujące hipotezy:
- im wcześniej robiono rezerwację, tym większa szansa anulacji (lead_time)
- powracający goście mogą rzadziej anulować rezerwacje (repeated_guest)
- segmenty kanałów sprzedaży mogą różnie wpływać na szansę anulacji, np.rezerwacje firmowe mogą być rzadziej odwoływane (market_segment_type)


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x='booking_status', y='lead_time')
plt.title('Lead time vs booking_status')
plt.show()

Im wcześniej robiono rezerwację, tym większa szansa na odwołanie.

In [ ]:
pd.crosstab(df["repeated_guest"], df["booking_status"], normalize="index") * 100

Powracający goście mają o wiele rzadziej odwołują rezerwacje.

In [ ]:
mseg = pd.crosstab(df["market_segment_type"], df["booking_status"], normalize="index") * 100
mseg = mseg.round(2)
display(mseg)
mseg.plot(kind="bar", stacked=True, figsize=(8,4))
plt.title("booking_status w grupach market_segment_type (w %)")
plt.xlabel("market_segment_type")
plt.ylabel("Procent")
plt.legend(title="booking_status")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

Rezerwacje robione online są najbardziej podatne na anulowanie. Najmniej odwołanych rezerwacji było w segmencie "Compemantary".

### Analiza rozkładów i zależności

In [ ]:
target_col = 'booking_status'
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols.remove(target_col)
print('Numeryczne:', num_cols)
print("Kategoryczne:", cat_cols)

In [ ]:
df.no_of_previous_bookings_not_canceled

### Najważniejsze zmienne numeryczne

In [ ]:
important_num = [
    'lead_time',
    'repeated_guest',
    'no_of_previous_cancellations',
    'no_of_previous_bookings_not_canceled',
    'avg_price_per_room',
    'no_of_week_nights',
    'no_of_weekend_nights',
    'no_of_special_requests'
]

fig, ax = plt.subplots(4, 2, figsize=(14, 14))

for i, col in enumerate(important_num):
    sns.histplot(df[col], bins=30, kde=True, ax=ax[i//2, i%2])
    ax[i//2, i%2].set_title(f'Rozkład: {col}')
    ax[i//2, i%2].set_xlabel(col)
    ax[i//2, i%2].set_ylabel('Liczba obserwacji')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(4, 2, figsize=(14, 14))

for i, col in enumerate(important_num):
    sns.boxplot(x=df[col], ax=ax[i//2, i%2])
    ax[i//2, i%2].set_title(f'Boxplot: {col}')
    ax[i//2, i%2].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x='repeated_guest')
plt.title("Rozkład repeated_guest")
plt.show() 
# jest to zmienna binarna, więc lepiej ją widać na wykresie słupkowym

In [ ]:
corr = df[important_num].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Macierz korelacji cech numerycznych')
plt.tight_layout()
plt.show()

Widać, że najbardziej skorelowane ze sobą cechy to:
- repeated_guest i no_of_previous_bookings_not_canceled
- repeated_guest i no_of_previous_cancellations
- no_of_previous_cancellations i no_of_previous_bookings_not_canceled
Jest to zgodne z intuicją - poracający goście częściej posiadają historię wcześniejszych rezerwacji. Pozostałe korelacje nie wskazują na istotane zależności liniowe.


### Najważniejsze zmienne kategoryczne

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))

important_cat = [
    "market_segment_type",
    "room_type_reserved",
    "type_of_meal_plan"
]

for i, col in enumerate(important_cat):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax[i])
    ax[i].set_title(f"Rozkład: {col}")
    ax[i].set_xlabel(col)
    ax[i].set_ylabel("Liczba")
    ax[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### Wnioski

Na podstawie powyższych wykresów, możemy zauważyć, że:
- większość klientów rezerwuje do ok. 3 miesięcy z wyprzedzeniem, ale część rezerwuje bardzo wcześnie, nawet ponad rok wcześniej (dużo wartości odstających)
- więcej wartości odstających przy braku anulowania rezerwacji, niż przy anulowaniu rezerwacji, czyli większość klientów nie ma historii anulacji
- cena pokojów ma bardzo dużo outlierów
- większość pobytów jest krótkich, zwłaszcza weekendowych
- większość gości nie zgłasza specjalnych próśb
- najwięcej rezerwacji było robionych online
- najczęściej był reerwowany pokój nr 1
- najczęściej był wybierany posiłek nr 1

# 2. Przetwarzanie danych

### Usunięcie kolumny Booking_ID

In [ ]:
df = df.drop('Booking_ID', axis = 1)
df.head()

### Kodowanie zmiennej docelowej

In [ ]:
target = {'Canceled': 1, 'Not_Canceled': 0} # Zmiana booking_status na liczbe
df['booking_status'] = df['booking_status'].replace(target)
print("Canceled = 1, Not_Canceled = 0")
df.describe()

### Podział danych na zbiór treningowy, walidacyjny i testowy w proporcjach 60/20/20

In [ ]:
X = df.drop('booking_status', axis = 1)
y = df['booking_status']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=123)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=123) # 0.25 * 0.80 = 0.20

### Kodowanie zmiennych kategorycznych

In [ ]:
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeryczne cechy:", num_features)
print("Kategoryczne cechy:", cat_features)

In [ ]:
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
data_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

data_pipeline.fit(X_train)

feature_names = (
    pd.Index(data_pipeline.named_steps['preprocessor'].get_feature_names_out())
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

X_train_processed = pd.DataFrame(data_pipeline.transform(X_train),
                                columns=feature_names)

X_val_processed = pd.DataFrame(data_pipeline.transform(X_val),
                                columns=feature_names)

X_test_processed = pd.DataFrame(data_pipeline.transform(X_test),
                                columns=feature_names)

# 3. Trening i ewaluacja modeli

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_v, y_v, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_v)
    y_prob = model.predict_proba(X_v)[:, 1]
    return {
        'Model': name,
        'Accuracy':  np.round(accuracy_score(y_v, y_pred), 4),
        'Precision': np.round(precision_score(y_v, y_pred), 4),
        'Recall':    np.round(recall_score(y_v, y_pred), 4),
        'F1-Score':  np.round(f1_score(y_v, y_pred), 4),
        'ROC-AUC':   np.round(roc_auc_score(y_v, y_prob), 4),
    }

results = []
trained_models = {}

### Regresja logistyczna

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=123)
res = evaluate_model(lr, X_train_processed, y_train, X_val_processed, y_val, 'Logistic Regression')
results.append(res)
trained_models['Logistic Regression'] = lr

for k, v in res.items():
    print(f"{k}: {v}")

### Drzewo decyzyjne

In [ ]:
dt = DecisionTreeClassifier(random_state=123)
res = evaluate_model(
    dt,
    X_train_processed, y_train,
    X_val_processed, y_val,
    'Decision Tree'
)

results.append(res)
trained_models['Decision Tree'] = dt

for k, v in res.items():
    print(f"{k}: {v}")

### Random forest

In [ ]:
rf = RandomForestClassifier(random_state=123)

res = evaluate_model(
    rf,
    X_train_processed, y_train,
    X_val_processed, y_val,
    'Random Forest'
)

results.append(res)
trained_models['Random Forest'] = rf

for k, v in res.items():
    print(f"{k}: {v}")

### SVC

In [ ]:
sv = SVC(random_state=123, probability=True)

res = evaluate_model(
    sv,
    X_train_processed, y_train,
    X_val_processed, y_val,
    'SVC'
)

results.append(res)
trained_models['SVC'] = sv

for k, v in res.items():
    print(f"{k}: {v}")

In [ ]:
param_dist = {
    'n_estimators': [100, 150, 200, 250, 300, 400],
    'max_depth': [10, 15, 20, 25, 30, 35, 40, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 3, 4, 5]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

print('Szukanie najlepszych parametrów...')
random_search.fit(X_train_processed, y_train)

print('Najlepsze znalezione parametry:', random_search.best_params_)

best_rf = random_search.best_estimator_
res_tuned = evaluate_model(best_rf, X_train_processed, y_train, X_val_processed, y_val, 'Tuned Random Forest')

results.append(res_tuned)
trained_models['Tuned Random Forest'] = best_rf

print('\nWyniki modelu po tuningu na zbiorze walidacyjnym:')
for k, v in res_tuned.items():
    print(f'{k}: {v}')

## Porównanie modeli

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
print("Porównanie modeli na zbiorze walidacyjnym")
print(results_df)

print("\nNajlepszy model wg. metryki")
for col in results_df.columns:
    best = results_df[col].idxmax()
    print(f"  {col:12s}: {best} ({results_df.loc[best, col]:.4f})")


## Wybór najlepszego modelu i ewaluacja na zbiorze testowym (todo)

In [ ]:
final_model = trained_models['Tuned Random Forest']

y_test_pred = final_model.predict(X_test_processed)
y_test_prob = final_model.predict_proba(X_test_processed)[:, 1]

print("Wyniki na zbiorze testowym")
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")

print(classification_report(y_test, y_test_pred))